In [ ]:
%run init_notebook.py
import torch, torch_directml

SRATE = 12000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch_directml.device()
print(f"Using device: {device}")


# Data Augmentation for Neural Sound Synthesis:

Soruce: https://github.com/iver56/torch-audiomentations

Data augmentation is used to increase the diversity of training data without actually collecting new data. 

### Common techniques include:
1. **Pitch Shifting**: Changing the pitch of the audio without affecting its duration.
2. **Time Stretching**: Changing the duration of the audio without affecting its pitch.
3. **Adding Reverb**: Simulating different acoustic environments by adding reverberation.
4. **Filtering**: Applying various filters (e.g., low-pass, high-pass) to alter the frequency content of the audio.
5. **Polarity Inversion**: Inverting the audio signal to create a new version of the sound.
6. **Polyphonic Mixing**: Combining multiple audio samples to create a new, more complex sound.
7. **Compression and Distortion**: Applying dynamic range compression or distortion effects to alter the sound characteristics.
8. **Combination of Techniques**: Using multiple augmentation techniques together to create even more diverse training data.


In [ ]:
from torch_audiomentations import Compose, PitchShift, LowPassFilter, HighPassFilter, BandPassFilter, PolarityInversion

mode = "per_example"  # aplicar una transformacion diferente a cada ejemplo del batch
p_mode = "per_example" # la probabilidad de aplicar cada transformacion se decide de forma independiente para cada ejemplo del batch
p = 0.5 # probabilidad de aplicar cada transformacion # TODO ver que valor poner aqui

# PitchShift
min_transpose_semitones = -4
max_transpose_semitones = 4
# LowPassFilter
min_cutoff_freq = 300.0
max_cutoff_freq = 6000.0
# HighPassFilter
min_cutoff_freq = 300.0
max_cutoff_freq = 6000.0
# BandPassFilter
min_center_freq = 300.0
max_center_freq = 6000.0
min_q_factor = 0.5
max_q_factor = 2.0

apply_augmentation = Compose(
    transforms=[
        PitchShift(mode=mode, p=p, p_mode=p_mode, sample_rate=SRATE, min_transpose_semitones=min_transpose_semitones, max_transpose_semitones=max_transpose_semitones),
        LowPassFilter(mode=mode, p=p, p_mode=p_mode, sample_rate=SRATE, min_cutoff_freq=min_cutoff_freq, max_cutoff_freq=max_cutoff_freq),
        HighPassFilter(mode=mode, p=p, p_mode=p_mode, sample_rate=SRATE, min_cutoff_freq=min_cutoff_freq, max_cutoff_freq=max_cutoff_freq),
        BandPassFilter(mode=mode, p=p, p_mode=p_mode, sample_rate=SRATE, min_center_frequency=min_center_freq, max_center_frequency=max_center_freq, min_bandwidth_fraction=min_q_factor, max_bandwidth_fraction=max_q_factor),
        PolarityInversion(mode=mode, p=p, p_mode=p_mode)
    ]
)

In [ ]:

# Make an example tensor with white noise.
# This tensor represents 8 audio snippets with 1 channel (mono) and 2 s of 16 kHz audio.
audio_samples = torch.rand(size=(8, 1, 32000), dtype=torch.float32, device=device) - 0.5

# Apply augmentation. This varies the gain and polarity of (some of)
# the audio snippets in the batch independently.
perturbed_audio_samples = apply_augmentation(audio_samples, sample_rate=SRATE)